In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/Setup/Utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://child-company-dp/{data_source}'
landing_path = f'{base_path}/landing'
processed_path = f'{base_path}/processed'
print("Base Path:", base_path)
print("Landing Path:", landing_path)
print("Processed Path:", processed_path)

bronze_table = f'{catalog}.{bronze_schema}.bronze_{data_source}'
silver_table = f'{catalog}.{silver_schema}.silver_{data_source}'
gold_table = f'{catalog}.{gold_schema}.gold_{data_source}'


Base Path: s3://child-company-dp/orders
Landing Path: s3://child-company-dp/orders/landing
Processed Path: s3://child-company-dp/orders/processed


In [0]:
df = spark.read.options(header=True, inferSchema=True).csv(f"{landing_path}/*.csv").withColumn("read_timestamp",F.current_timestamp()).select("*","_metadata.file_name","_metadata.file_size")
print("Total Rows: ", df.count())

Total Rows:  51810


In [0]:
display(df.limit(20))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FOCT62720602,"Tuesday, September 30, 2025",ABC987,25891301,71.0,2026-07-18T17:16:40.242Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",789720,25891502,125.0,2026-07-18T17:16:40.242Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",789720,25891403,462.0,2026-07-18T17:16:40.242Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",INVALID,25891601,133.0,2026-07-18T17:16:40.242Z,orders_2025_09_30.csv,41446
FOCT62720602,"Tuesday, September 30, 2025",789720,25891602,79.0,2026-07-18T17:16:40.242Z,orders_2025_09_30.csv,41446
FOCT62720602,2025/09/30,XYZ123,25891101,472.0,2026-07-18T17:16:40.242Z,orders_2025_09_30.csv,41446
FOCT62720602,2025/09/30,ABC987,25891301,71.0,2026-07-18T17:16:40.242Z,orders_2025_09_30.csv,41446
FOCT62402303,"Tuesday, September 30, 2025",789402,25891303,24.0,2026-07-18T17:16:40.242Z,orders_2025_09_30.csv,41446
FOCT62402303,"Tuesday, September 30, 2025",789402,25891301,26.0,2026-07-18T17:16:40.242Z,orders_2025_09_30.csv,41446
FOCT62402303,"Tuesday, September 30, 2025",789402,25891102,328.0,2026-07-18T17:16:40.242Z,orders_2025_09_30.csv,41446


In [0]:
df.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- order_placement_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_qty: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true") \
    .mode("append") \
    .saveAsTable(bronze_table)


In [0]:
files = dbutils.fs.ls(landing_path)
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

In [0]:
df_orders = spark.sql(f"SELECT * FROM {bronze_table}")
df_orders.show()


+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2026-07-18 17:29:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2026-07-18 17:29:...|orders_2025_07_01...|    20744|
|FJUL33320501|          01-07-2025|     789320|  25891403|     NULL|2026-07-18 17:29:...|orders_2025_07_01...|    20744|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891201|    354.0|2026-07-18 17:29:...|orders_2025_07_01...|    20744|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891501|    249.0|2026-07-18 17:29:...|orders_2025_07_01...|    20744|
|FJUL33320501|          01-07-20

In [0]:
df_orders = df_orders.filter(F.col("order_qty").isNotNull())


df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
    .otherwise("999999")
    .cast("string")
)
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"),r"^[A-Za-z]+,\s*","")
)

df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date" , "yyyy/MM/dd"),
        F.try_to_date("order_placement_date" , "dd-MM-yyyy"),
        F.try_to_date("order_placement_date" , "dd/MM/yyyy"),
        F.try_to_date("order_placement_date" , "MMMM dd, yyyy"),
        
    )
)

df_orders = df_orders.dropDuplicates(["order_id","order_placement_date","customer_id","product_id","order_qty"])

df_orders = df_orders.withColumn("product_id" , F.col('product_id').cast("string"))
df_orders.show()

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FJUL33320501|          2025-07-01|     789320|  25891203|    150.0|2026-07-18 17:29:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025-07-01|     789320|  25891301|     46.0|2026-07-18 17:29:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025-07-01|     789320|  25891201|    354.0|2026-07-18 17:29:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025-07-01|     789320|  25891501|    249.0|2026-07-18 17:29:...|orders_2025_07_01...|    20744|
|FJUL33401603|          2025-07-01|     789401|  25891302|     40.0|2026-07-18 17:29:...|orders_2025_07_01...|    20744|
|FJUL33401603|          2025-07-

In [0]:
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-07-01|2025-11-30|
+----------+----------+



In [0]:
display(df_orders.limit(20))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FJUL33320501,2025-07-01,789320,25891203,150.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891301,46.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891201,354.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744
FJUL33320501,2025-07-01,789320,25891501,249.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891302,40.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891502,133.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891503,145.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891203,429.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744
FJUL33401603,2025-07-01,789401,25891201,461.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744
FJUL32101601,2025-07-01,789101,25891503,183.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744


In [0]:
df_products = spark.table("fmcg.silver.products")

display(df_products.limit(5))

product_code,division,category,product,variant,product_id,read_timestamp,file_name,file_size
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,Healthy Snacks,Healthy Snacks,SportsBar Oats Cookie Bites ChocoChip (180g),180g,25891503,2026-07-18T15:40:29.246Z,products.csv,1388
e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,Nutrition Bars,Energy Bars,SportsBar Energy Bar Choco Fudge (40g),40g,25891102,2026-07-18T15:40:29.246Z,products.csv,1388
d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (300g),300g,25891302,2026-07-18T15:40:29.246Z,products.csv,1388
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,Dairy & Recovery,Recovery Dairy,SportsBar Greek Yogurt Pro Vanilla (80g),80g,25891403,2026-07-18T15:40:29.246Z,products.csv,1388
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,Breakfast Foods,Granola & Cereals,SportsBar Granola Crunch Honey Almond (400g),400g,25891301,2026-07-18T15:40:29.246Z,products.csv,1388


In [0]:
df_joined = df_orders.join(df_products,on = "product_id", how = "inner").select(df_orders["*"], df_products["product_code"])
display(df_joined.limit(10))

order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size,product_code
FJUL32202603,2025-07-01,789202,25891103,370.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268
FJUL33321602,2025-07-01,789321,25891602,74.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6
FJUL34303602,2025-07-01,789303,25891102,317.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e
FJUL34622602,2025-07-01,789622,25891402,439.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744,fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49
FJUL33402303,2025-07-01,789402,25891303,63.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f
FJUL33721603,2025-07-01,999999,25891602,148.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6
FJUL32420602,2025-07-01,789420,25891602,76.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6
FJUL33201602,2025-07-01,789201,25891602,52.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6
FJUL33403501,2025-07-01,789403,25891101,379.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843
FJUL33421203,2025-07-01,789421,25891203,99.0,2026-07-18T17:29:44.212Z,orders_2025_07_01.csv,20744,889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2


In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed","true"
    ).option("mergeSchema","true").mode("overwrite").saveAsTable(silver_table)

else:
    silver_delta = DeltaTable.forName(spark,silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"),"silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute() 


In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {silver_table};")
df_gold.show(2)


+------------+----------+-------------+--------------------+----------+-------------+
|    order_id|      date|customer_code|        product_code|product_id|sold_quantity|
+------------+----------+-------------+--------------------+----------+-------------+
|FJUL32202603|2025-07-01|       789202|102628255d24304d6...|  25891103|        370.0|
|FJUL33321602|2025-07-01|       789321|778c2a7aa27bfdb21...|  25891602|         74.0|
+------------+----------+-------------+--------------------+----------+-------------+
only showing top 2 rows


In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table"),
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema","true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"),"source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


creating New Table


In [0]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
df_child.show(10)

+----------+--------------------+-------------+-------------+
|      date|        product_code|customer_code|sold_quantity|
+----------+--------------------+-------------+-------------+
|2025-07-01|102628255d24304d6...|       789202|        370.0|
|2025-07-01|778c2a7aa27bfdb21...|       789321|         74.0|
|2025-07-01|e92c739a8d78cd6cb...|       789303|        317.0|
|2025-07-01|fe5a8036be4b9a787...|       789622|        439.0|
|2025-07-01|c68834ceaff15846b...|       789402|         63.0|
|2025-07-01|778c2a7aa27bfdb21...|       999999|        148.0|
|2025-07-01|778c2a7aa27bfdb21...|       789420|         76.0|
|2025-07-01|778c2a7aa27bfdb21...|       789201|         52.0|
|2025-07-01|e91ba9d665f90254d...|       789403|        379.0|
|2025-07-01|889c67757ece9c973...|       789421|         99.0|
+----------+--------------------+-------------+-------------+
only showing top 10 rows


In [0]:
df_monthly = (
    df_child
   # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
   .withColumn("month_start", F.trunc("date", "MM"))
   # or F.date_trunc("month", "date").cast("date")
   .groupBy("month_start", "product_code", "customer_code")
   .agg(F.sum("sold_quantity").alias("sold_quantity"))
   .withColumnRenamed("month_start", "date")
)
display(df_monthly.limit(10))


date,product_code,customer_code,sold_quantity
2025-07-01,102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,789202,2944.0
2025-07-01,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,789321,1742.0
2025-07-01,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e,789303,4929.0
2025-07-01,fe5a8036be4b9a787b7c0ae013fc752a8cfb6c55a2f7b2fd152a6380925e9c49,789622,3810.0
2025-07-01,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f,789402,960.0
2025-07-01,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,999999,6938.0
2025-07-01,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,789420,1565.0
2025-07-01,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,789201,1691.0
2025-07-01,e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843,789403,5872.0
2025-07-01,889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2,789421,3318.0


In [0]:
df_monthly.count()

3060

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]